# Tree–Cluster–Forest Subgroup Pipeline

This notebook runs a two-stage pipeline:

1. **Module 1 – Y-only clustering-based subgroup discovery**
   - Clusters the target variable `Y` using hierarchical clustering over many cut levels.
   - Each Y-cluster is a *candidate* subgroup.
   - Candidates are scored with a size-corrected KL objective and a diversity term (SYFLOW-style).
   - The top `K` subgroups are selected.

2. **Module 2 – Random forest explanations (Colab-style)**
   - For each selected subgroup, trains many random forests to predict subgroup membership from `X`.
   - Selects a forest that is accurate and simple (short decision paths).
   - For each subgroup, prints a Colab-like explanation tree (text and optional plot).

At the end there is an optional **SYFLOW-style visualization cell**, plotting the target distribution with different colours for each subgroup, annotated with subgroup rules.


In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.tree import export_text, plot_tree

from src.config_subgroups import TreeClusterForestConfig
from src.data_module import load_dataset
from src.cluster_subgroups import (
    discover_subgroups_by_clustering,
    build_subgroup_column,
)
from src.forest_explainer import (
    train_forest_for_subgroup,
    evaluate_forest,
)

# Optional: SYFLOW-style plotting utility (if available)
try:
    from src.demo_utils import plot_subgroups
    HAS_PLOT_SUBGROUPS = True
except ImportError:
    HAS_PLOT_SUBGROUPS = False
    print("Warning: src.demo_utils.plot_subgroups not found; SYFLOW-style plot will use a simple fallback.")


In [2]:
def show_explanation_tree_for_subgroup(model, feature_names, subgroup_index=None):
    """
    Show a Colab-style explanation tree for a single subgroup:
      - print the tree as text (export_text)
      - plot the tree (plot_tree)
    """
    # pick a representative tree from the forest (first estimator)
    best_tree = model.estimators_[0]

    # text output (Colab-style)
    header = (
        f"\nText tree explanation for subgroup {subgroup_index}"
        if subgroup_index is not None
        else "\nText tree explanation"
    )
    print(header)
    tree_text = export_text(best_tree, feature_names=feature_names)
    print(tree_text)

    # graphical tree
    plt.figure(figsize=(10, 6))
    plot_tree(
        best_tree,
        feature_names=feature_names,
        class_names=["not in subgroup", "in subgroup"],
        filled=True,
        rounded=True,
        fontsize=8,
    )
    if subgroup_index is not None:
        plt.title(f"Explanation tree for subgroup {subgroup_index}")
    else:
        plt.title("Explanation tree")
    plt.show()


In [3]:
def run_pipeline_on_dataset(
    dataset_name,
    cfg,
    save_csv=True,
    output_dir="outputs_subgroups",
):
    print(f"\n=== DATASET: {dataset_name} ===")

    # ------------------------------------------------------------------
    # Load data
    # ------------------------------------------------------------------
    X, Y, feature_names, is_clf = load_dataset(dataset_name)
    n, d = X.shape
    print(f"X shape = {X.shape}, Y shape = {Y.shape}, classification={is_clf}")

    # ------------------------------------------------------------------
    # MODULE 1: Y-only clustering-based subgroup discovery
    # ------------------------------------------------------------------
    masks, rules_box, scores = discover_subgroups_by_clustering(
        X,
        Y,
        cfg,
        feature_names=feature_names,
        n_subgroups=cfg.n_subgroups,
    )

    n_subgroups_found = len(masks)
    print(f"Discovered {n_subgroups_found} subgroups.")

    if n_subgroups_found == 0:
        print("No valid subgroups found; returning early.")
        return {
            "X": X,
            "Y": Y,
            "feature_names": feature_names,
            "masks": [],
            "rules_box": [],
            "scores": [],
            "df": None,
        }

    for i, (rule, score) in enumerate(zip(rules_box, scores)):
        print(f"\n[Module 1] Subgroup {i}:")
        print(f"  score = {score:.4f}")
        print(f"  box rule = {rule}")

    # Build a single subgroup_id column for the CSV
    subgroup_id = build_subgroup_column(
        masks,
        n_samples=n,
        fill_value=-1,
    )

    df = pd.DataFrame(X, columns=feature_names)
    df["target"] = Y
    df["subgroup_id"] = subgroup_id

    if save_csv:
        os.makedirs(output_dir, exist_ok=True)
        out_path = os.path.join(output_dir, f"{dataset_name}_with_subgroups.csv")
        df.to_csv(out_path, index=False)
        print(f"\nSaved dataset with subgroup_id to: {out_path}")

    # ------------------------------------------------------------------
    # MODULE 2: Random-forest explanations (Colab-style)
    # ------------------------------------------------------------------
    print("\n=== MODULE 2: Random-forest explanations ===")
    for k, mask in enumerate(masks):
        sg_size = int(mask.sum())
        print(f"\n--- Subgroup {k} (size={sg_size}) ---")
        model, rf_candidates = train_forest_for_subgroup(X, mask, cfg)

        if model is None:
            print("Skipping: subgroup too small to train a stable forest.")
            continue

        acc, comp = evaluate_forest(model, X, mask)
        print(f"Selected forest: accuracy={acc:.3f}, avg path length={comp:.2f}")

        # Colab-style explanation tree output
        show_explanation_tree_for_subgroup(model, feature_names, subgroup_index=k)

    # Return all artefacts for optional plotting / further analysis
    return {
        "X": X,
        "Y": Y,
        "feature_names": feature_names,
        "masks": masks,
        "rules_box": rules_box,
        "scores": scores,
        "df": df,
    }


In [ ]:
cfg = TreeClusterForestConfig()
print("Datasets to process:", cfg.datasets)

results = {}

for name in cfg.datasets:
    res = run_pipeline_on_dataset(name, cfg, save_csv=True, output_dir="outputs_subgroups")
    results[name] = res


Datasets to process: ['california']

=== DATASET: california ===
X shape = (20640, 8), Y shape = (20640,), classification=False


In [ ]:
# Choose one dataset to visualize (first in the config list by default)
dataset_name = cfg.datasets[0]
res = results[dataset_name]

X = res["X"]
Y = res["Y"]
masks = res["masks"]
rules_box = res["rules_box"]
feature_names = res["feature_names"]

if len(masks) == 0:
    print(f"No subgroups found for dataset '{dataset_name}', skipping SYFLOW-style plot.")
else:
    # Pick which subgroups to show (e.g., up to the first 3)
    which_sgs = list(range(min(3, len(masks))))

    print(f"SYFLOW-style target histograms for dataset '{dataset_name}', subgroups {which_sgs}")

    if HAS_PLOT_SUBGROUPS:
        # Use your existing helper if available
        plot_subgroups(
            target=Y,
            subgroups=masks,
            rules=rules_box,   # you can replace with RF-derived rules later
            which_sgs=which_sgs,
        )
    else:
        # Simple fallback: overlay histograms manually
        plt.figure(figsize=(8, 5))
        for idx in which_sgs:
            sg = Y[masks[idx]]
            plt.hist(sg, bins=50, alpha=0.5, label=f"SG {idx}: {rules_box[idx]}")
        plt.xlabel("Target Y")
        plt.ylabel("Count")
        plt.title(f"SYFLOW-style subgroup histograms: {dataset_name}")
        plt.legend()
        plt.show()
